CONVERSIONI DI COLORE:BGR, RGB E LO SPAZIO HSV

I colori sono solo numeri, ma il modo in cui ordiniamo questi numeri può cambiare il senso di tutto ciò che la macchine vede.

Conversione di colore: BGR, RGB, e lo spazio HSV

- Il paradosso di Open CV
- spazio HSV
- segmentazione cromatica

L'ordine dei canali
Il mistero del formato BGR
Standard storici e interoperabilità nel 2026
Quando carichiamo un'immaine con OpenCV, i canali sono ordinati come Blu, Verde, Rosso (BGR), a differenza dello standard universale RGB
Questa sceta, se ignorata, porta a visualizzazioni errate dove i visi appaiono bluastri e i cieli rossastri compromettendo l'analisi visiva.

Ma perchè un software così moderno si porta dietro questa strana convenzione?

Gestione dei Canali e Visualizzazione
Dal sensore allo schermo
* Open CV adotta lordine BGR per ragioni storiche legate ai produttori di fotocamere e agli standard video originali. Allora era il formato standard. ora è complicato cambiare.
* La funzione cv2.cvColor è il ponte necessario per trasformare i dati tra diversi spazi colore in modo efficiente.
* Librerie come Matplotlib e PIL si aspettano il formato RGB, rendendo obbligatoria la conversione prima della visualizzazione
* Manipolare i canali tramite slicing NumPy è possile ma meno performante rispetto alle funzioni native di OpenCV (scritte in c++ ed ottimizzate per essere molto veloci)

Errore comuni e soluzioni
L'errore classico è quello di passare un immagine BGR ad una libreria che si aspetta RBG e il rosso ed il blu finisco per scambiarsi il posto. Questo non è solo un problema estetico, ma potrebbe portare a predizioni errate poichè le feature spaziali sarebbero alterate.
Per l'ai un cielo rosso ed un prato blu sono segni di un mondo che non esiste.

A livello di bit stiamo cambiando l'ordine dei piani di informazione, spostare gli inquilini del primo piano all'ultimo e viceversa.
Esistono oltre 150 metodi di conversione in Open CV, ognuno ottimizzato per diverse architetture hardware.

I formato RGB o BGR hanno un limie, la luce.
immagina di riconoscere una mela rossa, in pieno sole è un rosso brillante, al tramonto quasi marrone, in ombra addirittura violacea. 
Nello spazio RBG se cambia la luce cambiano tutti e 3 i valori numerici e diventa un vero incubo matematico per le reti.
Ecco perchè introduciamo lo spazio HSV qui distinguiamo il cosa (il colore) dal quanto (la luce). E' come avere un iterrutore per il colore ed uno separato per la luminosità

Hue, Saturation e Value
Le tre dimensioni del colore
* Hue rappresenta la tonalità pura ed è misurata in gradi, mappare su un intervallo 0-179 in OpenCV. E' come una ruota panoramica di colori con valori/calori da 0 a 179
* Saturation indica l'intensità o la purezza del colore, dove 0 è grigio e 255 è il colore vivido.  indica quanto il colore è puro, una maglietta nuova è molto satura, una sbiadita dal solo ha saturazione bassa.
* Value descrive la luminisità: questo canale ci permette di rendere l'algoritmo robusto alle variazioni di luce. Il value è il valore massimo tra i canali RGB, se è 0 l'immagine è nera, indipendentemente dal colore scelto 
* La separazione di questi componenti semplifica enormemente l'identificazione di oggetti sotto luci diverse.

Perchè questa separazione è la chiave per rendere l'ai immune alle ombre?

Analisi del modello HSV
Vantaggio nel Tracking
In HSV, se un oggetto rosso finisce in ombra, cambieraà la sua Saturation e il suo Value, ma la sua Hue rimarrà costante. Se una macchina gialla finisce sotto un ponte, cambia Saturation e Value ma Hu (giallo) rimarrà stabile.
Usando HSV possiamo scrivere algoritmo che riconoscono un oggetto anche se si muove tra zone di luce e zone di ombra. Ma attenzione, se l'oggetto è quasi grigio o nero la Hue diventa instabile, è come cercare di capire la direzione del vento quando non c'è aria.

Ma matemaricamente come si passa dal cubo RGB a questo nuovo spazio cilindrico HSV?

Trasformazioni non lineari
Dall'array cubito al cono
Mentre RGB è un cubo, HSV è matematicamente rappresentabile come un cilindro o  un cono.
Il calcolo della Saturazione dipende dal rapporta tra il valore massimo e minimo dei canali RGB originali.
S= (max(RGB)-min(RGB)/max(RGB))
Più c'è differenza tra il colore più forte e quello più debole, più il colore è vibrante. Capire questo è fondamentale soprattutto quando progettiamo sistemi di visione che devono funzionare sia in una fabbrica illuminata al neon sia a cielo aperto sotto le nuvole.

Come usiamo questo colore perfetto?

Segmentazione tramite Maschere
La segmentazione è l'arte di separare l'informazione dal rumore, consiste nel crare un'immagine binaria dove solo i pixel di interesse sono bianchi, tutto il resto nero.
Questo processo utilizza la funzione cv2.in Range per definire i confini del colore cercato nello spazio HSV
Supponiamo di avere una foto di un prato con fiori rossi, se vuoi che l'ai veda i fiori, devi creare una maschera, immagine binaria, bianco dove c'è quello che interessa, nero tutto il resto.

Come impostare questi confini tramite la maschera
Dobbiamo definire un limite inferiore e uno superiore. Non cerchiamo un singolo numero, ma un volume di colore nello spazio HSV 
Diciamo alla macchina prendi tutto cioò che è abbastanza giallo, saturo e luminoso, il risultato è una mappa di bit, dove se condizione è vera abbiamo 255 (bianco) se è falsa abbiamo 0 (nero).

Ma come troviamo questi numeri magici per i limiti inferiore e superiore?

Workflow di Colore Detection
Trovare il bound non è semplice, usiamo il trackbar interattive per calibrare Hue, Saturation e Value. 
Un caso particolare: Il rosso si trova alle due estremità del range Hue (0-10 e 170-179), richiedendo spesso l'unione di due maschere
Il rosso è l'unico colore che si spezza, devo quindi creare due maschere
Spesso le maschere contengono rumore che deve essere rimosso con operazioni morfologiche, che vedremo aventi.

Una volta ottenuta la maschere come la applichiamo
Usiamo la logica binaria, and tra l'immagine e la maschera.



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import requests

# Configurazione ambiente 2026: Impostiamo Keras 3 con backend PyTorch
os.environ["KERAS_BACKEND"] = "torch"
import keras

def color_space_analysis_from_url(url: str):
    """
    Scarica un'immagine da un URL, esegue la segmentazione HSV e 
    converte il risultato in un tensore per il Deep Learning.
    """
    
    # --- 1. CARICAMENTO DA INTERNET ---
    print(f"[*] Download in corso da: {url}")
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        
        # Trasformiamo la risposta in un array di byte e decodifichiamo
        # IMREAD_COLOR forza il caricamento in formato BGR (Blue-Green-Red)
        image_array = np.frombuffer(response.content, np.uint8)
        img_bgr = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
        
        if img_bgr is None:
            raise ValueError("Errore: Impossibile decodificare l'immagine.")
    except Exception as e:
        print(f"[-] Errore durante il recupero dell'immagine: {e}")
        return None

    # --- 2. CONVERSIONE PER VISUALIZZAZIONE (RGB) ---
    # Swap dei canali per Matplotlib: BGR \rightarrow RGB
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # --- 3. PASSAGGIO ALLO SPAZIO HSV ---
    # Fondamentale per il tracking. 
    # Teoria: In RGB, un'ombra cambia tutti e 3 i canali contemporaneamente.
    # In HSV, il colore è isolato nel canale 'H'.
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # --- 4. SEGMENTAZIONE: CREAZIONE DI UNA MASCHERA ---
    # Obiettivo: Isolare oggetti di colore Verde.
    # OpenCV HSV: Hue [0-179], Saturation [0-255], Value [0-255].
    lower_green = np.array([35, 50, 50])   
    upper_green = np.array([85, 255, 255]) 

    # Operazione binaria: M(x,y) = 255 se il pixel è nel range, altrimenti 0.
    mask = cv2.inRange(img_hsv, lower_green, upper_green)

    # --- 5. OPERAZIONE BITWISE ---
    # Estraiamo i pixel verdi dall'immagine RGB originale usando la maschera binaria.
    res = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

    # --- 6. INTEGRAZIONE KERAS 3 ---
    # Normalizzazione a a[0, 1] e conversione in tensore compatibile con PyTorch.
    tensor_res = keras.ops.convert_to_tensor(res, dtype="float32") / 255.0

    # Visualizzazione didattica
    plt.figure(figsize=(15, 5))
    titles = ['Originale (RGB)', 'Maschera Binaria', 'Segmentazione Verde']
    images = [img_rgb, mask, res]

    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.imshow(images[i], cmap='gray' if i==1 else None)
        plt.title(titles[i], fontweight='bold')
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

    return tensor_res

# Esecuzione con un'immagine di prova (Paesaggio con molto verde)
if __name__ == "__main__":
    target_url = "https://picsum.photos/id/191/800/600" # Foto di natura
    color_space_analysis_from_url(target_url)